# DrawNet Training Pipeline

This notebook covers the training of **DrawNet**, which is responsible for detecting **Dyslexia** (reversals) and **Dysgraphia** (motor delays/kinematic errors) from touchscreen stroke data and letter drawings.

## Modality & Scope
- Input: Letter drawings and stroke coordinates
- Output: Multi-class probabilities (0 = Normal, 1 = Reversal, 2 = Corrected)
- Base Network: **EfficientNetB0** transfer learning + fine-tuning

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install dependencies if not present in Colab
!pip install -q tensorflow mediapipe opencv-python pandas numpy matplotlib kaggle

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, applications

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Step 2: Kaggle Credentials Setup
Upload your `kaggle.json` file to Google Colab. The code below will copy it to the correct location and configure permissions.

In [ ]:
from google.colab import files

kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')
if not os.path.exists(kaggle_path):
    print("Please upload your kaggle.json token now:")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    for filename in uploaded.keys():
        os.rename(filename, kaggle_path)
    os.chmod(kaggle_path, 0o600)
    print("Kaggle API configured successfully.")
else:
    print("Kaggle API token already present.")

## Step 3: Dataset Acquisition
We download the two datasets designated in `docs/dataset-map.md` for handwriting CNN pre-training and Hindi script norm pre-training:
1. `drizasazanitaisa/dyslexia-handwriting-dataset` (Dyslexia Handwriting reversals/corrected)
2. `suvooo/hindi-character-recognition` (Devanagari Deployed Script characters)

In [ ]:
# Create directories
os.makedirs('dataset/dyslexia', exist_ok=True)
os.makedirs('dataset/devanagari', exist_ok=True)
os.makedirs('output', exist_ok=True)

# Download Dyslexia Handwriting Dataset
!kaggle datasets download -d drizasazanitaisa/dyslexia-handwriting-dataset -p dataset/dyslexia
with zipfile.ZipFile('dataset/dyslexia/dyslexia-handwriting-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset/dyslexia')

# Download Devanagari Character Recognition Dataset
!kaggle datasets download -d suvooo/hindi-character-recognition -p dataset/devanagari
with zipfile.ZipFile('dataset/devanagari/hindi-character-recognition.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset/devanagari')

print("Datasets successfully downloaded and extracted.")

## Step 4: Preprocessing & Data Generators
We resize all handwriting inputs to `224x224` to fit the EfficientNetB0 input shape, apply rotations (+-10 degrees), and augment brightness values (+-0.2) to simulate variable classroom lighting. Flip options are disabled to preserve letter-reversal structures.

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

# Data augmentation pipeline for transfer training
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(factor=0.03), # approx +- 10 degrees
    layers.RandomBrightness(factor=0.2),
    # Note: DO NOT add horizontal or vertical flip. It would break letter reversal indicators!
])

# We will use Keras directory generators. Adjust directory names depending on the extracted structures.
# For pre-training Devanagari Character Norms:
devanagari_dir = 'dataset/devanagari/DevanagariHandwrittenCharacterDataset/Train'
if os.path.exists(devanagari_dir):
    train_ds_devanagari = tf.keras.utils.image_dataset_from_directory(
        devanagari_dir,
        validation_split=0.2,
        subset="training",
        seed=42,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE
    )
    val_ds_devanagari = tf.keras.utils.image_dataset_from_directory(
        devanagari_dir,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE
    )

## Step 5: Pre-train on Devanagari Script (Critical for Indian Classroom Deployment)
We train an EfficientNetB0 structure on the Devanagari characters first. This step validates that the base network adjusts to non-Latin stroke characteristics before learning the specific dyslexia categories.

In [ ]:
base_model = applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze pre-trained weights

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(46, activation='softmax')(x) # 46 classes in Devanagari dataset

devanagari_model = tf.keras.Model(inputs, outputs)
devanagari_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fast pre-training run (2 epochs for validation)
if os.path.exists(devanagari_dir):
    print("Pre-training base model on Devanagari writing...")
    devanagari_model.fit(train_ds_devanagari, validation_data=val_ds_devanagari, epochs=2)

## Step 6: Fine-tune on Dyslexia Handwriting Reversals
Now we switch the final classification head to target the 3 dyslexia subclasses: Normal (0), Reversal (1), Corrected (2).

In [ ]:
# Load dyslexia handwriting dataset structure
dyslexia_dir = 'dataset/dyslexia/dataset'

train_ds_dys = tf.keras.utils.image_dataset_from_directory(
    dyslexia_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)
val_ds_dys = tf.keras.utils.image_dataset_from_directory(
    dyslexia_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

# Create final classification structure
base_features = devanagari_model.layers[-3].output # Get output of Pooling layer
new_fc = layers.Dense(128, activation='relu')(base_features)
new_out = layers.Dense(3, activation='softmax')(new_fc) # 3 outputs (Normal, Reversal, Corrected)

drawnet_model = tf.keras.Model(devanagari_model.input, new_out)
drawnet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# First train the classification head
print("Training classifier heads...")
history = drawnet_model.fit(train_ds_dys, validation_data=val_ds_dys, epochs=3)

# Unfreeze base layers for fine tuning
base_model.trainable = True
drawnet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Fine-tuning entire network...")
history_fine = drawnet_model.fit(train_ds_dys, validation_data=val_ds_dys, epochs=2)

## Step 7: Export to TFLite (Float16 Quantized)
Finally, we serialize the model to TFLite format with Float16 quantization to guarantee memory footprints under 50MB and local inference latencies under 15ms.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(drawnet_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

output_path = 'output/seren_drawnet.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model successfully exported to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")